In [ ]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [ ]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [ ]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [ ]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

## EXAMPLES OF BYTE COMPARISONS

In [ ]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [ ]:
# no letters in common
w1b & w2b

In [ ]:
# letters in common
w1b & w3b

In [ ]:
# bitwise or
w1b | w2b

In [ ]:
# this is the same as directly above
byte_encode_words('abhorcleft')

# BUILD LEVEL 2

In [ ]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

In [ ]:
l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [ ]:
l2_array = l2_df['l2'].to_numpy(dtype = np.int32)

# CREATE ARRAYS OF THE SAME SHAPE

In [ ]:
N_SAMPLE_SIZE = 10000
L3_N_SAMPLE_SIZE = 50000

l2_array_test = l2_array[:20000]
l2_cut_range = range(0, l2_array.shape[0] + N_SAMPLE_SIZE, N_SAMPLE_SIZE)

# number of words/word bytes
n_wba = word_byte_array.shape[0]

# this the word as bytes array
r_wba = np.repeat(a = np.reshape(word_byte_array, shape = (1, n_wba)),
                  repeats = N_SAMPLE_SIZE, axis = 0)

l5_output = []

for i_l2_idx, l2_idx in enumerate(l2_cut_range[1:]):
    pre_l2_idx = l2_cut_range[i_l2_idx]
    
    
    # sample some rows!
    # work with N_SAMPLE_SIZE
    tl2 = l2_array[pre_l2_idx:l2_idx]     

    # I want each value to be repeated 5977 times across. In the x direction
    # these are the l2 combos
    r_tl2 = np.reshape(np.repeat(a = tl2, repeats = n_wba), shape = (tl2.shape[0], n_wba))
    
    # # compare two columns
    # (r_tl2[:, 1] == r_tl2[:, 100]).all()
    # # compare two rows
    # (r_wba[0, :] == r_wba[10, :]).all()

    # bitwise AND, check when the values are 0, meaning there are shared letters, and then convert the booleans to integers
    l2_and = ((r_tl2 & r_wba) == 0).astype(np.int32)
    # bitwise OR, multiply by the 1/0 matrix. This will indicate groups of three words that do not share a common letter.
    l2_or = (r_tl2 | r_wba) * l2_and
    
    
    # get the indices...
    i, j = np.where(l2_or > 0)
    # these are the indices of the word as bytes
        
    # we can also use the i,j args in the r_tl2 and r_wba matrices
    # l2_or is the level 3 word group
    # how do put this into a new matrix?
    l3_array = l2_or[i, j]
    print('l3:', l2_idx,  l3_array.shape)
    
    # bop that into the same shape
    # I want each value to be repeated 5977 times across. In the x direction
    l3_cut_range = range(0, l3_array.shape[0] + L3_N_SAMPLE_SIZE, L3_N_SAMPLE_SIZE)

    for i_l3_idx, l3_idx in enumerate(l3_cut_range[1:]):
        pre_l3_idx = l3_cut_range[i_l3_idx]

        #print('l3', l3_idx)
    
        # sample some rows!
        # work with N_SAMPLE_SIZE
        tl3 = l3_array[pre_l3_idx:l3_idx]        

        r_tl3 = np.reshape(np.repeat(a = tl3, repeats = n_wba), shape = (tl3.shape[0], n_wba))

        if tl3.shape != N_SAMPLE_SIZE:
            r_wba_l3 = np.repeat(a = np.reshape(word_byte_array, shape = (1, n_wba)), repeats = tl3.shape[0], axis = 0)
        else:
            r_wba_l3 = r_wba.copy()
    
        l3_and = ((r_tl3 & r_wba_l3) == 0).astype(np.int32)
        l3_or = (r_tl3 | r_wba_l3) * l3_and
        # get the indices...
        i, j = np.where(l3_or > 0)
        
        # this is l4        
        l4_array = l3_or[i, j]
        print('l4:', l2_idx, l3_idx, l4_array.shape)
    
        # bop that into the same shape
        # I want each value to be repeated 5977 times across. In the x direction
        l4_cut_range = range(0, l4_array.shape[0] + L3_N_SAMPLE_SIZE, L3_N_SAMPLE_SIZE)

        for i_l4_idx, l4_idx in enumerate(l4_cut_range[1:]):
            pre_l4_idx = l4_cut_range[i_l4_idx]
    
            # sample some rows!
            # work with N_SAMPLE_SIZE
            tl4 = l4_array[pre_l4_idx:l4_idx]     

            r_tl4t = np.reshape(np.repeat(a = tl4, repeats = n_wba),
                                shape = (tl4.shape[0], n_wba))
            r_wba_l4 = np.repeat(a = np.reshape(word_byte_array, shape = (1, n_wba)),
                                 repeats = tl4.shape[0], axis = 0)

            # AND
            l4_and = ((r_tl4t & r_wba_l4) == 0).astype(np.int32)

            # OR
            l4_or = (r_tl4t | r_wba_l4) * l4_and

            i, j = np.where(l4_or > 0)

            l5_array = l4_or[i, j]
            print('l5:', l2_idx, l3_idx, l4_idx, l5_array.shape)

            if l5_array.size > 0:
                l5_output.append(l5_array)
